# Notebook 4: Skills-Based Agents and A2A Evaluation

This notebook demonstrates the **Skills-based architecture** for AI agents, inspired by Claude's Agent Skills pattern.

## Skills Architecture Overview

Instead of hardcoded tools, Skills-based agents use **modular capabilities** defined in markdown files:

```
.claude/skills/
├── universe-selection/SKILL.md    # Asset universe selection
├── optimization-execution/SKILL.md # MVO/HRP optimization
├── risk-assessment/SKILL.md       # Risk metrics evaluation
├── backtesting/SKILL.md           # Historical validation
└── portfolio-comparison/SKILL.md  # Compare alternatives
```

## Key Benefits

1. **Modularity**: Each capability is self-contained with instructions and examples
2. **Progressive Disclosure**: Only load skill details when needed
3. **Easy Updates**: Modify skills without changing agent code
4. **Testability**: Each skill can be tested independently

## Comparison: Tools vs Skills

| Aspect | Tools-based Agent | Skills-based Agent |
|--------|-------------------|--------------------|
| Definition | Python functions | Markdown files |
| Instructions | In code/docstrings | Rich markdown with examples |
| Updates | Requires code changes | Edit markdown files |
| Flexibility | Fixed parameters | Flexible execution |
| Testing | Unit tests | Skill evaluation |

## Part 1: Setup and Configuration

In [1]:
import os
import json
import warnings
from typing import Dict, Any, List
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
load_dotenv(dotenv_path='../.env')

# Configure Gemini API (for Green agent)
import google.generativeai as genai
gemini_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if gemini_api_key:
    genai.configure(api_key=gemini_api_key)
    os.environ["GOOGLE_API_KEY"] = gemini_api_key

# Verify Anthropic API key (for Skills agent)
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

# Import Skills-based agent components
from agents import (
    # Skills-based agents (LangChain wrapper - for comparison)
    SkillLoader,
    Skill,
    create_skills_purple_agent,
    run_skills_agent,
    run_skills_a2a_evaluation,
    # Native Anthropic Skills agents (recommended)
    create_native_skills_agent,
    run_native_skills_agent,
    run_native_skills_a2a_evaluation,
    # Standard A2A components (Green agent stays the same)
    create_a2a_green_agent,
    create_rag_knowledge_base,
    A2AEvaluation,
    A2AMessage,
    flush_langfuse,
    LLMProvider
)

# Load data
with open('scenarios.json', 'r') as f:
    SCENARIOS = json.load(f)
with open('evaluation_dataset.json', 'r') as f:
    EVAL_DATA = json.load(f)

# Provider configuration
SKILLS_PROVIDER = LLMProvider.ANTHROPIC  # Skills agent uses Claude
GREEN_PROVIDER = LLMProvider.GEMINI      # Green agent uses Gemini

warnings.filterwarnings('ignore')

print(f"Anthropic API configured: {'Yes' if anthropic_api_key else 'No'}")
print(f"Gemini API configured: {'Yes' if gemini_api_key else 'No'}")
print("\nSkills-based Agent Architecture")
print("================================")
print(f"Purple Agent: Native Anthropic Skills ({SKILLS_PROVIDER.value})")
print(f"Green Agent: Standard ({GREEN_PROVIDER.value})")

Anthropic API configured: Yes
Gemini API configured: Yes

Skills-based Agent Architecture
Purple Agent: Native Anthropic Skills (anthropic)
Green Agent: Standard (gemini)


## Part 2: Exploring Skills

Let's examine the Skills that are available to our agent.

In [2]:
# Load and inspect Skills
skill_loader = SkillLoader()

print(f"Loaded {len(skill_loader.skills)} Skills from .claude/skills/\n")
print("="*60)
print("AVAILABLE SKILLS")
print("="*60)

for skill in skill_loader.get_all_skills():
    print(f"\n[{skill.name}]")
    print(f"  Description: {skill.description[:80]}...")
    print(f"  Path: {skill.path}")

Loaded 5 Skills from .claude/skills/

AVAILABLE SKILLS

[backtesting]
  Description: Backtest portfolio performance using historical data. Use when validating portfo...
  Path: /Users/oleksandrhonchar/Documents/GitHub/iskpi-workshops-ai-evaluation/claude-code-work/.claude/skills/backtesting/SKILL.md

[portfolio-comparison]
  Description: Compare different portfolio optimization approaches and present trade-offs. Use ...
  Path: /Users/oleksandrhonchar/Documents/GitHub/iskpi-workshops-ai-evaluation/claude-code-work/.claude/skills/portfolio-comparison/SKILL.md

[optimization-execution]
  Description: Execute portfolio optimization using Mean-Variance Optimization (MVO) or Hierarc...
  Path: /Users/oleksandrhonchar/Documents/GitHub/iskpi-workshops-ai-evaluation/claude-code-work/.claude/skills/optimization-execution/SKILL.md

[risk-assessment]
  Description: Assess portfolio risk using volatility, drawdown, and other risk metrics. Use wh...
  Path: /Users/oleksandrhonchar/Documents/GitHub/

In [3]:
# View a specific Skill's instructions (progressive disclosure)
skill_name = "optimization-execution"
instructions = skill_loader.get_skill_instructions(skill_name)

print(f"SKILL: {skill_name}")
print("="*60)
print(instructions[:2000] + "..." if len(instructions) > 2000 else instructions)

SKILL: optimization-execution
# Portfolio Optimization Skill

## Optimization Methods

### Mean-Variance Optimization (MVO)

Classic Markowitz optimization with multiple targets:

| Target | Description | Best For |
|--------|-------------|----------|
| `min_volatility` | Minimize portfolio variance | Conservative investors |
| `max_sharpe` | Maximize risk-adjusted returns | Balanced investors |
| `max_return` | Maximize expected return | Aggressive investors |
| `efficient_return` | Target return with min risk | Specific return goals |

### Hierarchical Risk Parity (HRP)

Machine learning-based allocation using hierarchical clustering:
- More robust to estimation errors
- Better diversification across asset clusters
- No need to invert covariance matrix

## When to Use Each Method

| Investor Profile | Recommended Method | Target |
|-----------------|-------------------|--------|
| Conservative, capital preservation | MVO | `min_volatility` |
| Balanced, risk-adjusted returns | MVO | 

## Part 3: Native Anthropic Skills Agent

Create the Native Skills-based Purple Agent using Anthropic SDK directly. This agent:

1. **Reads skill files** via bash: `cat .claude/skills/optimization-execution/SKILL.md`
2. **Executes Python code** via bash: `python3 -c "from portfolio_optimizer import ..."`
3. **Follows skill instructions** to complete tasks

This approach is more aligned with Anthropic's Skills architecture than wrapping skills in LangChain tools.

In [4]:
# Create Native Anthropic Skills Agent
native_skills_agent = create_native_skills_agent()

print(f"Native Skills Agent created!")
print(f"Model: {native_skills_agent['model']}")
print(f"Skills Directory: {native_skills_agent['skill_loader'].skills_dir}")
print(f"\nNative Tools:")
print("  - bash: Execute commands (read skills, run Python code)")
print("  - str_replace_editor: File operations")

Native Skills Agent created!
Model: claude-sonnet-4-20250514
Skills Directory: /Users/oleksandrhonchar/Documents/GitHub/iskpi-workshops-ai-evaluation/claude-code-work/.claude/skills

Native Tools:
  - bash: Execute commands (read skills, run Python code)
  - str_replace_editor: File operations


In [5]:
# Test Native Skills agent with a simple request
print("Testing Native Anthropic Skills Agent...")
print("="*60)

test_result = run_native_skills_agent(
    native_skills_agent,
    "What skills do you have available? List them briefly.",
    session_id="notebook4_native_test"
)

print("\nAgent Response:")
print(test_result["output"])

Testing Native Anthropic Skills Agent...

Agent Response:
Based on the tools available to me, I have the following skills:

## Portfolio Optimization Agent Skills

1. **Portfolio Analysis & Optimization**
   - Analyze investor profiles (age, goals, risk tolerance)
   - Generate optimized portfolios using modern portfolio theory
   - Choose from 3 investment universes: conservative, global diversified, or US tech
   - Optimize for minimum volatility or maximum Sharpe ratio

2. **Investment Universe Selection**
   - **Conservative**: Low-risk assets for capital preservation
   - **Global Diversified**: Balanced mix for moderate risk/growth  
   - **US Tech**: High-growth technology stocks for aggressive investors

3. **Portfolio Backtesting & Metrics**
   - Calculate expected returns, volatility, and Sharpe ratios
   - Perform historical backtests with drawdown analysis
   - Generate detailed asset allocation recommendations

4. **File Management**
   - View, create, and edit files
   - 

## Part 4: Native Skills Agent in Action

Let's see how the Native Skills agent handles a portfolio request by reading skill files and executing code.

In [6]:
# Test with an investor persona
sarah = SCENARIOS['personas']['sarah_conservative']

print(f"Portfolio Request for {sarah['name']}")
print("="*60)
print(f"Narrative: {sarah['narrative'][:200]}...")
print("="*60)

result = run_native_skills_agent(
    native_skills_agent,
    sarah['narrative'],
    session_id="notebook4_native_sarah"
)

print("\n" + "="*60)
print("NATIVE SKILLS AGENT RESPONSE")
print("="*60)
print(result["output"][:2000] + "..." if len(result["output"]) > 2000 else result["output"])

Portfolio Request for Sarah Chen
Narrative: I'm Sarah, 58 years old, and planning to retire in 7 years. I have $500,000 saved and want stable income with capital preservation. I can't afford to lose more than 15% of my portfolio. I prefer US-ba...
  [Tool: bash]
  [Tool: str_replace_editor]
  [Tool: bash]
  [Tool: bash]
  [Tool: bash]
  [Tool: str_replace_editor]
  [Tool: str_replace_editor]
  [Tool: bash]
  [Tool: str_replace_editor]
  [Tool: bash]

NATIVE SKILLS AGENT RESPONSE


[Note: Agent reached iteration limit. Results may be incomplete.]


## Part 5: Green Agent (Evaluator)

The Green Agent stays the same as in Notebook 3. It has:
- `search_knowledge_base` - RAG retrieval
- `web_search` - DuckDuckGo web search

The Green Agent will evaluate the Skills-based Purple Agent.

In [7]:
# Create Green Agent (same as Notebook 3, uses Gemini)
retriever = create_rag_knowledge_base(EVAL_DATA)
green_agent = create_a2a_green_agent(retriever, provider=GREEN_PROVIDER)

print(f"Green Agent (Evaluator) created with {GREEN_PROVIDER.value}")
print("Tools: search_knowledge_base (RAG), web_search (DuckDuckGo)")

Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring


Green Agent (Evaluator) created with gemini
Tools: search_knowledge_base (RAG), web_search (DuckDuckGo)


## Part 6: A2A Evaluation of Native Skills Agent

Now we run the A2A protocol with:
- **Purple Agent**: Native Anthropic Skills agent (uses bash to read skills + execute code)
- **Green Agent**: Standard evaluator with RAG + web search

The Green agent will evaluate how well the Native Skills agent:
1. Reads and follows skill instructions
2. Executes code correctly via bash
3. Provides a complete and accurate recommendation

In [8]:
# Run A2A evaluation with Native Skills-based Purple Agent
elena = SCENARIOS['personas']['elena_balanced']

print(f"A2A EVALUATION: Native Skills Agent")
print(f"Investor: {elena['name']}")
print("Protocol: GREEN (Standard Evaluator) <-> PURPLE (Native Skills)")
print("Rounds: 3")
print("="*60)

native_a2a_result = run_native_skills_a2a_evaluation(
    task_description=elena['narrative'],
    native_agent_config=native_skills_agent,
    evaluator_agent=green_agent,
    session_id="notebook4_native_a2a",
    max_rounds=3
)

print("\n" + "="*60)
print("A2A EVALUATION RESULTS")
print("="*60)
print(f"\nTotal messages: {len(native_a2a_result.conversation)}")
print(f"Overall Score: {native_a2a_result.overall_score:.1f}/10")

print("\nDimension Scores:")
for dim, score in native_a2a_result.scores.items():
    print(f"  {dim}: {score:.1f}/10")

A2A EVALUATION: Native Skills Agent
Investor: Elena Rodriguez
Protocol: GREEN (Standard Evaluator) <-> PURPLE (Native Skills)
Rounds: 3

NATIVE SKILLS A2A PROTOCOL - ROUND 1: Initial Request

[GREEN → PURPLE (Native Skills)] Please provide a portfolio recommendation for this investor: I'm Elena, 38 years old, and I want to build a portfolio for my children's education fund. My kids are 2 and 5 years old, so I have about 1...
  [Tool: bash]


I0000 00:00:1768497016.648071 29885873 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


  [Tool: str_replace_editor]
  [Tool: bash]


I0000 00:00:1768497022.821321 29885873 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


  [Tool: bash]


...

  [Tool: bash]


I0000 00:00:1768497169.355191 29885873 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers


  [Tool: bash]


I0000 00:00:1768497177.961065 29885873 fork_posix.cc:71] Other threads are currently calling into gRPC, skipping fork() handlers
Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



[PURPLE (Native Skills) → GREEN] Response received (67 chars)

NATIVE SKILLS A2A PROTOCOL - FINAL ASSESSMENT
Okay, I've reviewed the Purple Agent's portfolio recommendation and the subsequent explanations. Here's my final assessment:

**Scores:**

*   **Universe Selection:** 5/10 - The universe is limited to a few ETFs, lacking broader diversification across asset classes and geographies.
*   **Optimization Method:** 4/10 - The agent reached the iteration limit, suggesting potential issues with the optimization process. The heavy gold allocation raises concerns about the effectiveness of the optimization.
*   **Risk Assessment:** 6/10 - The agent provides risk metrics like volatility, Sharpe ratio, and maximum drawdown, but the risk assessment seems incomplete given the concentrated portfolio.
*   **Constraint Handling:** 8/10 - The agent respected the constraint of no single position exceeding 15%, although this was not difficult with only three assets.
*   **Explanation Quality:** 7

In [9]:
# Display conversation history
print("A2A CONVERSATION HISTORY")
print("="*60)

for i, msg in enumerate(native_a2a_result.conversation):
    agent_type = "Native Skills" if msg.sender == "purple" else "Standard"
    sender_label = f"GREEN (Evaluator)" if msg.sender == "green" else f"PURPLE ({agent_type})"
    print(f"\n[{i+1}] {sender_label} - {msg.message_type.upper()}")
    print("-" * 40)
    content = msg.content[:600] + "..." if len(msg.content) > 600 else msg.content
    print(content)

A2A CONVERSATION HISTORY

[1] GREEN (Evaluator) - REQUEST
----------------------------------------
Please provide a portfolio recommendation for this investor: I'm Elena, 38 years old, and I want to build a portfolio for my children's education fund. My kids are 2 and 5 years old, so I have about 18 years. I have $150,000 to invest with moderate risk tolerance. I'd like global diversification and reasonable risk-adjusted returns. No single position should exceed 15% of the portfolio.

[2] PURPLE (Native Skills) - RESPONSE
----------------------------------------


[Note: Agent reached iteration limit. Results may be incomplete.]

[3] GREEN (Evaluator) - QUERY
----------------------------------------
Okay, I see that the Purple Agent reached an iteration limit and the results might be incomplete. Before I can properly evaluate, I need more information about the proposed portfolio.

My follow-up question is: Can you please provide details on the asset allocation, including the specific a

## Part 7: Comparison - Native Skills vs Tools-based

Let's compare the Native Skills agent with the traditional tools-based agent from Notebook 3.

In [10]:
from agents import create_a2a_purple_agent, run_a2a_evaluation

# Create traditional Tools-based Purple Agent (uses Gemini)
tools_purple_agent = create_a2a_purple_agent(provider=GREEN_PROVIDER)

print("COMPARISON: Native Skills vs Tools Architecture")
print("="*60)
print(f"\nNative Skills Agent (Anthropic {native_skills_agent['model']}):")
print("  - bash: Read skill files, execute Python code")
print("  - str_replace_editor: File operations")
print(f"\nSkills Available (read via bash):")
for skill in native_skills_agent['skill_loader'].get_all_skills():
    print(f"  - {skill.name}")

print(f"\nTools-based Agent ({GREEN_PROVIDER.value}):")
for tool in tools_purple_agent.tools:
    print(f"  - {tool.name}")

Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'default' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring
Key 'title' is not supported in schema, ignoring


COMPARISON: Native Skills vs Tools Architecture

Native Skills Agent (Anthropic claude-sonnet-4-20250514):
  - bash: Read skill files, execute Python code
  - str_replace_editor: File operations

Skills Available (read via bash):
  - backtesting
  - portfolio-comparison
  - optimization-execution
  - risk-assessment
  - universe-selection

Tools-based Agent (gemini):
  - get_available_universes
  - optimize_portfolio_tool
  - run_hrp_optimization
  - compare_portfolios


In [ ]:
# Run A2A evaluation with Tools-based agent for comparison
print("Running A2A evaluation with Tools-based Purple Agent...")
print("="*60)

tools_a2a_result = run_a2a_evaluation(
    task_description=elena['narrative'],
    portfolio_agent=tools_purple_agent,
    evaluator_agent=green_agent,
    session_id="notebook4_tools_comparison",
    max_rounds=3
)

print("\n" + "="*60)
print("COMPARISON RESULTS")
print("="*60)

print(f"\n{'Metric':<25} {'Tools-Based':>15} {'Native Skills':>15}")
print("-" * 55)
print(f"{'Overall Score':<25} {tools_a2a_result.overall_score:>14.1f}/10 {native_a2a_result.overall_score:>14.1f}/10")
print(f"{'Messages':<25} {len(tools_a2a_result.conversation):>15} {len(native_a2a_result.conversation):>15}")

# Compare individual dimensions if available
all_dims = set(tools_a2a_result.scores.keys()) | set(native_a2a_result.scores.keys())
for dim in sorted(all_dims):
    tools_score = tools_a2a_result.scores.get(dim, '-')
    native_score = native_a2a_result.scores.get(dim, '-')
    tools_str = f"{tools_score:.1f}" if isinstance(tools_score, float) else tools_score
    native_str = f"{native_score:.1f}" if isinstance(native_score, float) else native_score
    print(f"{dim:<25} {tools_str:>15} {native_str:>15}")

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")


Running A2A evaluation with Tools-based Purple Agent...

A2A PROTOCOL - ROUND 1: Initial Request

[GREEN → PURPLE] Please provide a portfolio recommendation for this investor: I'm Elena, 38 years old, and I want to build a portfolio for my children's education fund. My kids are 2 and 5 years old, so I have about 1...

Invoking: `optimize_portfolio_tool` with `{'max_position': 0.15, 'universe': 'global', 'optimization_target': 'max_sharpe'}`
responded: Okay, Elena. Given your investment goals, time horizon, and risk tolerance, let's construct a portfolio for your children's education fund.

**1. Understand your goals:**

*   **Goal:** Build a portfolio for your children's education fund.
*   **Time Horizon:** Approximately 18 years (long-term).
*   **Risk Tolerance:** Moderate.
*   **Investment Amount:** $150,000.
*   **Preferences:** Global diversification, reasonable risk-adjusted returns, and no single position exceeding 15% of the portfolio.

**2. Select an appropriate universe:**



## Part 8: Key Takeaways

### Native Skills Architecture (Anthropic SDK)

The Native Skills approach uses Anthropic's SDK directly with bash and text_editor tools:

```python
from agents import create_native_skills_agent, run_native_skills_agent

# Create native agent
native_agent = create_native_skills_agent()

# Agent uses bash to read skill files
# cat .claude/skills/optimization-execution/SKILL.md

# Agent executes Python via bash
# python3 -c "from portfolio_optimizer import ..."

result = run_native_skills_agent(native_agent, "Build a portfolio")
```

### SKILL.md Format

Skills are markdown files with instructions:

```markdown
---
name: optimization-execution
description: Execute portfolio optimization using MVO or HRP...
---

# Portfolio Optimization Skill

## Instructions
[Detailed instructions with code examples...]
```

### How Native Skills Works

1. **System prompt** includes skill metadata (names + descriptions)
2. **Claude reads** skill instructions via bash: `cat .claude/skills/*/SKILL.md`
3. **Claude executes** Python code via bash: `python3 -c "..."`
4. **State persists** within the agent's conversation

### Architecture Comparison

| Aspect | Tools-based | Native Skills |
|--------|-------------|---------------|
| LLM | Any (Gemini, etc.) | Anthropic Claude |
| Tool calls | Function calls | bash + text_editor |
| Instructions | In prompts/docstrings | SKILL.md files |
| Code execution | Wrapper functions | Direct Python via bash |
| State | Per-tool isolation | Persistent in conversation |

### When to Use Each

| Use Case | Recommended |
|----------|-------------|
| Simple, well-defined tools | Tools-based |
| Complex domain expertise | Native Skills |
| Need rich instructions with examples | Native Skills |
| Flexible code execution | Native Skills |
| Must use non-Anthropic LLM | Tools-based |

In [ ]:
# Flush Langfuse events
flush_langfuse()
print("Langfuse events flushed!")
print("\nCheck Langfuse dashboard for:")
print("  - Skills-based agent traces")
print("  - A2A evaluation comparisons")
print("  - Sessions: notebook4_test, notebook4_sarah, notebook4_a2a")